In [1]:
import numpy as np
# import pandas as pd
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer


In [2]:
import os
for dirname, _, filenames in os.walk('./'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./notebook.ipynb
./embeddings.npy
./song_lyrics_preprocessed.parquet
./.git/config
./.git/HEAD
./.git/description
./.git/index
./.git/COMMIT_EDITMSG
./.git/objects/0e/1471c89879f1ef2dcfd8ae2d75eb0c760b6026
./.git/objects/02/ca16cf683dbfacb084f23835c6002cbecb1cec
./.git/objects/b5/8b603fea78041071d125a30db58d79b3d49217
./.git/objects/d7/53a22996e3d5ab38c1f7a20d02a62da70bc18a
./.git/objects/c0/10c6881297017691347921678bbc7c8fa5a902
./.git/objects/4e/a72a911af2d782f20b992e808276bb838718d8
./.git/objects/pack/tmp_pack_ODu8F0
./.git/objects/1f/2ea11e7f0069b02b6f715f9735b8a31b234538
./.git/objects/7e/f04e2ea0752989be4a246d998b53adf46bab42
./.git/objects/10/5ce2da2d6447d11dfe32bfb846c3d5b199fc99
./.git/objects/86/48f9401aa827dc53e658ac3b1c1f1556df236b
./.git/objects/ba/aa1589d00b76bc2294f86897207f522f111761
./.git/objects/e9/ccd44fe4c1f0a7f395d6e8c35d2b8fa68dcc6d
./.git/objects/1d/4c45ca918cf1e2d7dc37c86ef6e4c8ab00aa96
./.git/info/exclude
./.git/logs/HEAD
./.git/logs/refs/heads/main
./.git/ho

In [3]:
n = 30_000

parquet_file = pq.ParquetFile('./song_lyrics_preprocessed.parquet')
df = next(parquet_file.iter_batches(batch_size=n)).to_pandas()
df = df[['id', 'title', 'artist', 'year', 'lyrics', 'language_cld3']]

In [4]:
df

,id,title,artist,year,lyrics,language_cld3
0,1,Killa Cam,Cam'ron,2004,"[Chorus: Opera Steve & Cam'ron]\nKilla Cam, Ki...",en
1,3,Can I Live,JAY-Z,1996,"[Produced by Irv Gotti]\n\n[Intro]\nYeah, hah,...",en
2,4,Forgive Me Father,Fabolous,2003,Maybe cause I'm eatin\nAnd these bastards fien...,en
3,5,Down and Out,Cam'ron,2004,[Produced by Kanye West and Brian Miller]\n\n[...,en
4,6,Fly In,Lil Wayne,2005,"[Intro]\nSo they ask me\n""Young boy\nWhat you ...",en
...,...,...,...,...,...,...
29995,31694,Electric Kingdom vocal version,Twilight 22,1983,Electric kingdom\n\nDeep in the city people li...,en
29996,31695,The Corruptors Execution,E-40,1999,[Pimp C]\nHold up..\n\nIt's the motherfuckin C...,en
29997,31696,Do You Wana Freak,Freak Brothers,1997,Chorus:\n\nDo you wanna freak?\nDo you wanna f...,en
29998,31697,Family Affair HOF Fam,Reservoir Dogs,2009,"[Verse One] [Big Pooh]\nSince '07 I said, ""Fuc...",en


In [5]:
def build_song_text_corpus(row):
    parts = [
        str(row['title']),
        str(row['artist']),
        str(row['year']),
        str([row['lyrics']])[:500]
    ]

    return ' '.join([p for p in parts if p != 'nan'])

df['text_corpus'] = df.apply(build_song_text_corpus, axis=1)
df

,id,title,artist,year,lyrics,language_cld3,text_corpus
0,1,Killa Cam,Cam'ron,2004,"[Chorus: Opera Steve & Cam'ron]\nKilla Cam, Ki...",en,Killa Cam Cam'ron 2004 ['[Chorus: Opera Steve ...
1,3,Can I Live,JAY-Z,1996,"[Produced by Irv Gotti]\n\n[Intro]\nYeah, hah,...",en,Can I Live JAY-Z 1996 ['[Produced by Irv Gotti...
2,4,Forgive Me Father,Fabolous,2003,Maybe cause I'm eatin\nAnd these bastards fien...,en,Forgive Me Father Fabolous 2003 ['Maybe cause ...
3,5,Down and Out,Cam'ron,2004,[Produced by Kanye West and Brian Miller]\n\n[...,en,Down and Out Cam'ron 2004 ['[Produced by Kanye...
4,6,Fly In,Lil Wayne,2005,"[Intro]\nSo they ask me\n""Young boy\nWhat you ...",en,Fly In Lil Wayne 2005 ['[Intro]\nSo they ask m...
...,...,...,...,...,...,...,...
29995,31694,Electric Kingdom vocal version,Twilight 22,1983,Electric kingdom\n\nDeep in the city people li...,en,Electric Kingdom vocal version Twilight 22 198...
29996,31695,The Corruptors Execution,E-40,1999,[Pimp C]\nHold up..\n\nIt's the motherfuckin C...,en,The Corruptors Execution E-40 1999 ['[Pimp C]\...
29997,31696,Do You Wana Freak,Freak Brothers,1997,Chorus:\n\nDo you wanna freak?\nDo you wanna f...,en,"Do You Wana Freak Freak Brothers 1997 [""Chorus..."
29998,31697,Family Affair HOF Fam,Reservoir Dogs,2009,"[Verse One] [Big Pooh]\nSince '07 I said, ""Fuc...",en,Family Affair HOF Fam Reservoir Dogs 2009 ['[V...


In [6]:
embeddings_path = './embeddings.npy'
model = SentenceTransformer('all-MiniLM-L6-v2')
if os.path.exists(embeddings_path):
    embeddings = np.load(embeddings_path)
    print(f"Loaded embeddings from {embeddings_path}")
else:
    embeddings = model.encode(
        df['text_corpus'].tolist(),
        batch_size=64,
        show_progress_bar=True
    )
    np.save('embeddings.npy', embeddings)
print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embeddings from ./embeddings.npy
Embeddings shape: (30000, 384)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

def get_mood(lyrics):
    score = analyzer.polarity_scores(str(lyrics))['compound']
    if score > 0.3: return 'happy'
    elif score < -0.3: return 'sad'
    else: return 'neutral'

df['mood'] = df['lyrics'].apply(get_mood)
df['mood'].value_counts()

mood
sad        17710
happy      11770
neutral      520
Name: count, dtype: int64

In [9]:
def infer_mood(query):
    score = analyzer.polarity_scores(query)['compound']
    if score > 0.3: return 'happy'
    elif score < -0.3: return 'sad'
    else: return 'neutral'

def recommend(query, top_k=7, filter_mood=True):
    query_vec = model.encode([query])
    scores = cosine_similarity(query_vec, embeddings)[0]

    results = df.copy()
    results['scores'] = scores

    if filter_mood:
        mood = infer_mood(query)
        if mood:
            results = results[results['mood'] == mood]

    return results.sort_values('scores', ascending=False).head(top_k)[['title', 'artist', 'year', 'mood', 'scores']]

print(recommend("energetic jazz"))

                                                  title  \
1015                                      Jazz Weve Got   
29999  Freedom Jazz Dance Evolution Of the Groove Remix   
27835           Lord Jazz Hit Me One Time Make it Funky   
17327          My Definition of a Boombastic Jazz Style   
12509                                    Bass Head Jazz   
29509                                        Jazz Track   
19465                                     Jazzys Groove   

                                 artist  year   mood    scores  
1015               A Tribe Called Quest  1991  happy  0.540265  
29999                       Miles Davis  2007  happy  0.535290  
27835          Lords of the Underground  1993  happy  0.512181  
17327                    Dream Warriors  1991  happy  0.496236  
12509                       CeeLo Green  2002  happy  0.495230  
29509                         MC Buzz B  2011  happy  0.489132  
19465  DJ Jazzy Jeff & The Fresh Prince  1989  happy  0.468422  
